In [2]:
import jax
import jax.numpy as jnp
from jax.tree_util import tree_flatten, tree_unflatten
import equinox as eqx
import numpy as np
from scipy.optimize import minimize
from typing import Callable

# Use the same SeriesRC model from the previous example
class SeriesRC(eqx.Module):
    R: jax.Array
    C: jax.Array

    def __init__(self, R_init: float, C_init: float):
        self.R = jnp.asarray(R_init)
        self.C = jnp.asarray(C_init)

    def __call__(self, freq: jax.Array) -> jax.Array:
        omega = 2 * jnp.pi * freq
        return jnp.sqrt(self.R**2 + (1 / (omega * self.C))**2)

    def to_cost_function(
        self, 
        x_features: jax.Array, 
        y_target: jax.Array
    ) -> Callable[[eqx.Module], jax.Array]:
        """
        This is the PURE JAX version of the cost function.
        It takes a PyTree of parameters and returns a JAX scalar.
        """
        _, static = eqx.partition(self, eqx.is_array)

        def cost_function(params: eqx.Module) -> jax.Array:
            model = eqx.combine(params, static)
            y_predicted = model(x_features)
            return jnp.mean((y_predicted - y_target)**2)

        return cost_function

# --- 1. Initial Setup ---

# Create mock data (same as before)
key = jax.random.PRNGKey(0)
true_R, true_C = 50.0, 1e-9
true_model = SeriesRC(true_R, true_C)
freqs = jnp.logspace(3, 9, 100)
y_true = true_model(freqs)
y_data = y_true + 0.5 * jax.random.normal(key, y_true.shape)

# Instantiate the model with an initial guess
initial_guess_model = SeriesRC(R_init=10.0, C_init=1e-12)

# Get the pure JAX cost function
jax_cost_func = initial_guess_model.to_cost_function(x_features=freqs, y_target=y_data)

# Partition the model to get the initial PyTree of parameters
initial_params, _ = eqx.partition(initial_guess_model, eqx.is_array)

# --- 2. Flatten the PyTree for SciPy ---

# Flatten the initial parameters to get a 1D vector and the "recipe"
flat_initial_params, treedef = tree_flatten(initial_params)

# SciPy's minimize function needs a NumPy array, not a list of JAX arrays
x0 = np.concatenate([np.asarray(p).ravel() for p in flat_initial_params])

# --- 3. Create the Wrapper for SciPy ---

def scipy_objective_wrapper(flat_params_np: np.ndarray) -> float:
    """
    This function is the bridge between SciPy and JAX.
    - It takes a flat NumPy array from SciPy.
    - It returns a single float that SciPy can minimize.
    """
    # Unflatten the 1D vector back into the original PyTree structure
    params_tree = tree_unflatten(treedef, jnp.asarray(flat_params_np))
    
    # Execute the pure JAX cost function with the reconstructed PyTree
    loss = jax_cost_func(params_tree)
    
    # Return a standard float for SciPy
    return float(loss)

# --- 4. Run the SciPy Optimizer ---

print("Starting SciPy optimization...")
result = minimize(
    scipy_objective_wrapper, 
    x0, 
    method='BFGS' # A common quasi-Newton method
)
print("Optimization finished!")

# --- 5. Unflatten the Final Result ---

# The optimal parameters are in result.x (a flat NumPy array)
flat_optimal_params = result.x

# Unflatten the final result back into a PyTree
optimal_params_tree = tree_unflatten(treedef, jnp.asarray(flat_optimal_params))

# Recombine with the static part to get the final, fitted model
final_model = eqx.combine(optimal_params_tree, eqx.partition(initial_guess_model, eqx.is_array)[1])

print(f"\nTrue parameters:      R={true_R:.2f}, C={true_C:.2e}")
print(f"Fitted parameters:    R={final_model.R.item():.2f}, C={final_model.C.item():.2e}")


### Optimization: Providing Gradients to SciPy

# The above works, but it can be slow because SciPy has to approximate the gradients numerically. Since we have JAX, we can provide the *exact* gradients to make the optimization much faster and more accurate.

# You just need to create a second wrapper for the gradient.

# ```python
# Create a function that computes the gradient of the JAX cost function
jax_grad_func = jax.grad(jax_cost_func)

def scipy_grad_wrapper(flat_params_np: np.ndarray) -> np.ndarray:
    """This wrapper computes the gradient for SciPy."""
    # Unflatten the parameters, just like before
    params_tree = tree_unflatten(treedef, jnp.asarray(flat_params_np))
    
    # Compute the gradient. The result is a PyTree of gradients.
    grad_tree = jax_grad_func(params_tree)
    
    # Flatten the gradient PyTree into a 1D vector
    flat_grads, _ = tree_flatten(grad_tree)
    
    # Concatenate and convert to a NumPy array for SciPy
    return np.concatenate([np.asarray(g).ravel() for g in flat_grads])

# Now you can run the optimizer with the 'jac' (Jacobian/gradient) argument
print("\nStarting SciPy optimization with gradients...")
result_with_grad = minimize(
    scipy_objective_wrapper,
    x0,
    method='BFGS',
    jac=scipy_grad_wrapper # Provide the exact gradient!
)

Starting SciPy optimization...
Optimization finished!

True parameters:      R=50.00, C=1.00e-09
Fitted parameters:    R=10.00, C=5.13e-05

Starting SciPy optimization with gradients...
